# Chapter 18 — Is the Model Actually the Problem?

**Book alignment:** Debugging AI From First Principles, Chapter 18

**Question this notebook isolates:** The ticket says "model failure." Do swap probes in
stack order — same-model × different-pipeline first, then same-pipeline × different-model,
then params — convict exactly one layer cheapest-first, with a **stop rule** that skips the
expensive probes once a cheap layer is convicted?

In [ ]:
def system(*, pipeline_ok, weights_rev, params_ok, doc_rank):
    """Pass count on a 10-case fixture, given the boundary configuration."""
    if not pipeline_ok:
        return 3
    if doc_rank > 5:            # 'lost in the middle': decisive section present but buried
        return 4
    if not params_ok:          # e.g. max_tokens cuts the answer mid-citation
        return 4
    return 9 if weights_rev == "A" else 10

BASE = dict(pipeline_ok=False, weights_rev="A", params_ok=True, doc_rank=8)

## 1. Baseline, and three hypotheses with numeric forecasts

In [ ]:
base = [system(**BASE) for _ in range(5)]
print("baseline pass count (5 trials):", base)
assert base[0] <= 4
print("FORECAST  H1 pipeline: repaired-context >= 8/10")
print("FORECAST  H2 weights : repaired-context <= 4/10  AND  rev-B on identical bytes >= 8/10")
print("FORECAST  H3 params  : deterministic params flip >= 8/10 on identical bytes+weights")

## 2. Probe 1 — same weights + revision, repair one pipeline element

In [ ]:
probe1 = [system(**{**BASE, "pipeline_ok": True, "doc_rank": 1}) for _ in range(5)]
print("repaired-pipeline pass count:", probe1)
assert min(probe1) >= 8
print("H1 convicted for this fixture. STOP RULE: probes 2 and 3 do not run.")
convicted, ran_well_probe, ran_param_probe = "pipeline", False, False
assert not ran_well_probe and not ran_param_probe

## 3. Why the stop rule matters — and why a confounded upgrade proves nothing

In [ ]:
# position alone, holding presence fixed: rank 8 fails, rank 1 passes -> never a weights failure
buried  = [system(pipeline_ok=True, weights_rev="A", params_ok=True, doc_rank=8) for _ in range(5)]
front   = [system(pipeline_ok=True, weights_rev="A", params_ok=True, doc_rank=1) for _ in range(5)]
assert max(buried) <= 4 < min(front)
print(f"same section, rank 8 -> {buried[0]}/10 ; rank 1 -> {front[0]}/10   (a pipeline variable)")

# the confounded 'fix': new model AND new pipeline in one move
confounded = system(pipeline_ok=True, weights_rev="B", params_ok=True, doc_rank=1)
print(f"\nnew-model + new-pipeline together: {confounded}/10 - improved, and attributes to nothing")
print("the demo passes; the diagnosis is UNKNOWN; the pipeline rot is now certified by a bad experiment")

## What we earned

"Model failure" is a hypothesis, not an observation. The attribution stack — pipeline →
params → weights → intent — is walked cheapest-first, one swapped layer per experiment, with
the weights (the most expensive layer) convicted last. Probe 1 repaired one pipeline element
on the frozen model and flipped 3/10 → 9/10, so the stop rule skipped the weights and param
probes entirely. A position-sensitive failure (rank 8 vs rank 1) was never a weights bug,
and a simultaneous model+pipeline swap attributes to nothing.

**Notebook 19 / Chapter 19** opens the largest pipeline suspect: the exact bytes and token
ids the model received.